In [1]:
import pandas as pd
import os

# Check what got attached
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/yogeshrayal/dataset/DSL-StrongPasswordData.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/yogeshrayal/dataset/DSL-StrongPasswordData.csv')
print(df.shape)
print(df.head())
print(df.columns.tolist())
print("Unique users:", df['subject'].nunique())


(20400, 34)
  subject  sessionIndex  rep  H.period  DD.period.t  UD.period.t     H.t  \
0    s002             1    1    0.1491       0.3979       0.2488  0.1069   
1    s002             1    2    0.1111       0.3451       0.2340  0.0694   
2    s002             1    3    0.1328       0.2072       0.0744  0.0731   
3    s002             1    4    0.1291       0.2515       0.1224  0.1059   
4    s002             1    5    0.1249       0.2317       0.1068  0.0895   

   DD.t.i  UD.t.i     H.i  ...     H.a  DD.a.n  UD.a.n     H.n  DD.n.l  \
0  0.1674  0.0605  0.1169  ...  0.1349  0.1484  0.0135  0.0932  0.3515   
1  0.1283  0.0589  0.0908  ...  0.1412  0.2558  0.1146  0.1146  0.2642   
2  0.1291  0.0560  0.0821  ...  0.1621  0.2332  0.0711  0.1172  0.2705   
3  0.2495  0.1436  0.1040  ...  0.1457  0.1629  0.0172  0.0866  0.2341   
4  0.1676  0.0781  0.0903  ...  0.1312  0.1582  0.0270  0.0884  0.2517   

   UD.n.l     H.l  DD.l.Return  UD.l.Return  H.Return  
0  0.2583  0.1338       0.3509

In [4]:
import numpy as np

# Select only timing features (drop metadata)
feature_cols = [col for col in df.columns if col not in ['subject', 'sessionIndex', 'rep']]

# Compute ratio features — each H value divided by next H value
# This is the core GhostID innovation — ratios survive speed changes
H_cols = [col for col in feature_cols if col.startswith('H.')]
DD_cols = [col for col in feature_cols if col.startswith('DD.')]
UD_cols = [col for col in feature_cols if col.startswith('UD.')]

print(f"H features: {len(H_cols)}")
print(f"DD features: {len(DD_cols)}")
print(f"UD features: {len(UD_cols)}")
print(f"Total features: {len(feature_cols)}")

# Create ratio features between consecutive H values
for i in range(len(H_cols) - 1):
    df[f'ratio_{H_cols[i]}_{H_cols[i+1]}'] = df[H_cols[i]] / (df[H_cols[i+1]] + 1e-8)

ratio_cols = [col for col in df.columns if col.startswith('ratio_')]
print(f"\nRatio features created: {len(ratio_cols)}")


H features: 11
DD features: 10
UD features: 10
Total features: 41

Ratio features created: 10


In [5]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode user labels
le = LabelEncoder()
df['user_id'] = le.fit_transform(df['subject'])

# Full feature set — raw + ratio
all_features = feature_cols + ratio_cols

# X and y
X = df[all_features].values
y = df['user_id'].values

print(f"Feature vector size: {X.shape[1]}")
print(f"Total samples: {X.shape[0]}")
print(f"Number of users: {len(np.unique(y))}")

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape}")
print(f"Test: {X_test.shape}")

Feature vector size: 51
Total samples: 20400
Number of users: 51

Train: (16320, 51)
Test: (4080, 51)
